# 第7章　保存・加载・推理・迁移学习・下一步

学会把训练好的模型**保存并复用**，以及**复用预训练模型（迁移学习）**。
最后给出"常见坑"清单和后续学习路线。

> **本笔记使用方法**
> - 从上到下依次运行单元格（Colab/Jupyter 都是 `Shift + Enter`）。
> - 代码**稍作修改、弄坏再修好**最能进步。每章末尾有练习。
> - 多数章节不需要 GPU。较重的章节（CNN）会说明用法。

In [ ]:
import torch
print("PyTorch version:", torch.__version__)
print("CUDA (GPU) available:", torch.cuda.is_available())

## 7-1. 模型的保存与加载（推荐 `state_dict`）

推荐**只保存权重（`state_dict`）**。模型结构用代码重新定义。

In [ ]:
import torch
import torch.nn as nn

model = nn.Sequential(nn.Linear(10, 32), nn.ReLU(), nn.Linear(32, 2))

# 保存（只存权重）
torch.save(model.state_dict(), "model_weights.pth")
print("已保存")

# 加载：先建相同结构，再灌入权重
model2 = nn.Sequential(nn.Linear(10, 32), nn.ReLU(), nn.Linear(32, 2))
model2.load_state_dict(torch.load("model_weights.pth"))
model2.eval()    # 切到推理模式
print("加载完成")

## 7-2. 推理（预测）的固定写法
上线预测时务必：**`model.eval()` ＋ `with torch.no_grad():`**。

In [ ]:
model2.eval()
sample = torch.randn(1, 10)         # 1条假输入
with torch.no_grad():
    logits = model2(sample)
    prob = torch.softmax(logits, dim=1)
    pred = logits.argmax(dim=1)
print("概率:", prob, " 预测类别:", pred.item())

## 7-3. 迁移学习（Transfer Learning）— 小数据的救星

以在 ImageNet 等海量数据上预训练好的模型为底座，**只把最后一层换成你的任务**。
比从零训练更快、小数据也能高精度。思路：

1. 加载预训练模型（作为优秀的特征提取器）。
2. 前半部分（特征提取）**冻结**（`requires_grad=False`）不训练。
3. 只把最后的分类层**换成你的类别数**再训练。

> 下面单元格首次会下载预训练权重（需要联网，推荐 Colab）。太重就只读不跑。

In [ ]:
from torchvision import models

# 加载预训练 ResNet18（指定 weights）
net = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# 1) 冻结特征提取部分（不计算梯度＝训练中不更新）
for p in net.parameters():
    p.requires_grad = False

# 2) 把最后的 fc 换成你的类别数（例如 5 种花）
num_classes = 5
net.fc = nn.Linear(net.fc.in_features, num_classes)   # 只有这里 requires_grad=True

# 训练对象只有换掉的 fc
trainable = [n for n, p in net.named_parameters() if p.requires_grad]
print("要训练的参数:", trainable)
# 之后用和第6章相同的训练循环（把 net.fc.parameters() 交给 optimizer）即可

## 7-4. 常见坑清单（收藏版）

| 症状 | 原因 | 对策 |
|---|---|---|
| 损失不下降 | 忘了 `optimizer.zero_grad()`／学习率不合适 | 检查五步，`lr` 从 1e-3 附近调 |
| `shape` 报错 | 维度不一致 | 各处 `print(x.shape)` |
| 分类精度上不去 | 自己给输出加了 `softmax` | `CrossEntropyLoss` 要喂**原始 logits** |
| `Expected ... cuda ... cpu` | 模型和数据设备不一致 | 两者都 `.to(device)` |
| 评估结果不稳 | 忘了 `model.eval()` / `no_grad()` | 推理前两者都加 |
| 显存不足(OOM) | 批太大／在累积梯度 | 调小 `batch_size`，推理用 `no_grad` |
| `loss` 变成 `nan` | 学习率太大／输入没归一化 | 调小 `lr`，输入 `Normalize` |
| 过拟合（只有 train 高） | 模型太复杂／数据太少 | 数据增强・`Dropout`・正则・增数据 |

## 7-5. 下一步（路线图）

**深入**
- 官方教程：https://pytorch.org/tutorials/ （先 "Learn the Basics" → "60 Minute Blitz"）
- 官方文档：https://pytorch.org/docs/stable/

**选方向**
- 图像：数据增强、ResNet/EfficientNet、`torchvision`、分割
- 自然语言・LLM：**Transformer** 原理 → Hugging Face `transformers`、微调
- 语音・时序：RNN/LSTM、`torchaudio`

**推荐教材**
- 书《Deep Learning with PyTorch》(Manning，作者免费提供 PDF)
- 视频：Andrej Karpathy「Neural Networks: Zero to Hero」(YouTube，从零手写自动微分)
- 免费书：Dive into Deep Learning (https://d2l.ai/) 的 PyTorch 版

**实用工具（熟练后）**
- 省去训练循环样板的 **PyTorch Lightning**
- 实验管理 **TensorBoard** / Weights & Biases
- 海量预训练模型库 **Hugging Face Hub**

## 练习 7
1. 把第4章做的模型保存→在另一个单元格加载→跑通预测。
2. 把第6章的 CNN 保存，在新运行时里加载并复现 MNIST 测试精度。
3. （进阶）用 `torchvision.datasets` 的小图像数据，把 7-3 的迁移学习完整跑一遍。

In [ ]:
# 在这里写你自己的代码并运行
